# Real-Time Weather Alerts Agent

In [17]:
import os
import requests
from typing import Optional, List, Dict, Any
import nest_asyncio

# Import ADK modules
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from vertexai.preview.reasoning_engines import AdkApp

# Enable nested event loops for Jupyter / Colab Enterprise
nest_asyncio.apply()

# Define API Keys
os.environ["GOOGLE_MAPS_API_KEY"] = "xxxxxxxxxxxx"
os.environ["GEMINI_API_KEY"] = "YOUR_GEMINI_API_KEY"
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"  # Required for LiteLLM GPT-4o target


# ==========================================
# 1. PEP 8 COMPLIANT TOOL DEFINITIONS
# ==========================================

def get_lat_lon(location_name: str) -> Optional[Dict[str, float]]:
    """Fetch the latitude and longitude for a given location using Google Maps Geocoding API.

    Args:
        location_name (str): The place name or address (e.g., 'Miami, FL' or 'Seattle, WA').

    Returns:
        Optional[Dict[str, float]]: A dictionary containing 'lat' and 'lon' keys,
        or None if geocoding fails or is unavailable.
    """
    api_key = os.environ.get("GOOGLE_MAPS_API_KEY")
    if not api_key:
        print("[Error] GOOGLE_MAPS_API_KEY environment variable missing.")
        return None

    url = f"https://maps.googleapis.com/maps/api/geocode/json?address={location_name}&key={api_key}"
    try:
        response = requests.get(url, timeout=10)
        res_data = response.json()
        if res_data.get("status") == "OK" and res_data.get("results"):
            location = res_data["results"][0]["geometry"]["location"]
            return {"lat": location["lat"], "lon": location["lng"]}
        return None
    except Exception as e:
        print(f"[Tool Error] Geocoding failed: {e}")
        return None


def get_nws_weather_alerts(lat: float, lon: float) -> Optional[List[Dict[str, Any]]]:
    """Fetch active weather alerts from the U.S. National Weather Service (NWS) API.

    Args:
        lat (float): Latitude of the location (e.g., 25.7617).
        lon (float): Longitude of the location (e.g., -80.1918).

    Returns:
        Optional[List[Dict[str, Any]]]: A list of alert property dictionaries containing
        event name, severity, urgency, and instructions. Returns None if data is
        unavailable or an error occurs.
    """
    # NOAA / NWS requires a custom User-Agent header
    headers = {"User-Agent": "(ADKWorkshopAgent/1.0, lab@example.com)"}
    url = f"https://api.weather.gov/alerts/active?point={lat},{lon}"
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            return None

        data = response.json()
        features = data.get("features", [])

        alerts = []
        for feat in features:
            props = feat.get("properties", {})
            alerts.append({
                "event": props.get("event"),
                "severity": props.get("severity"),
                "urgency": props.get("urgency"),
                "headline": props.get("headline"),
                "instruction": props.get("instruction")
            })
        return alerts
    except Exception as e:
        print(f"[Tool Error] NWS lookup failed: {e}")
        return None


# ==========================================
# 2. AGENT CONFIGURATION (Gemini & Multi-Vendor)
# ==========================================

WEATHER_INSTRUCTIONS = """
You are Pat, an emergency weather copilot.
1. Use the 'get_lat_lon' tool to find latitude and longitude for the location requested.
2. Use 'get_nws_weather_alerts' to fetch active severe warnings for those coordinates.
3. If active warnings exist, summarize the event, severity level, urgency, and safety steps.
4. If no alerts exist, report that weather condition alerts are currently CLEAR.
"""

# Agent 1: Default Gemini Flash Model
weather_agent_gemini = Agent(
    name="Pat_Gemini",
    model="gemini-2.5-flash",
    description="Pat the Weather Agent (Gemini Engine).",
    instruction=WEATHER_INSTRUCTIONS,
    tools=[get_lat_lon, get_nws_weather_alerts]
)

# Agent 2: Third-Party Model (GPT-4o via LiteLLM)
weather_agent_gpt = Agent(
    name="Pat_GPT4o",
    model=LiteLlm(model="openai/gpt-4o"),
    description="Pat the Weather Agent (GPT-4o Engine).",
    instruction=WEATHER_INSTRUCTIONS,
    tools=[get_lat_lon, get_nws_weather_alerts]
)


# ==========================================
# 3. MULTI-CITY TEST RUNNER (Using AdkApp)
# ==========================================

def test_weather_agent(agent_instance: Agent, cities: List[str]):
    """Creates an AdkApp session and streams queries across multiple US cities."""
    print(f"\n================ TESTING AGENT: {agent_instance.name} ================")

    # Wrap agent inside an AdkApp container
    app = AdkApp(agent=agent_instance, enable_tracing=False)
    user_id = "workshop_tester"

    # Create user session (returns a dict)
    session = app.create_session(user_id=user_id)
    session_id = session.get("id") or session.get("name")

    for city in cities:
        print(f"\n--- Location Check: {city} ---")
        prompt = f"Check for active severe weather warnings in {city} and summarize safety steps."

        # Stream response via user session using extracted session_id
        for event in app.stream_query(
            user_id=user_id,
            session_id=session_id,
            message=prompt
        ):
            # Safe dict/object parsing for event output
            if isinstance(event, dict) and "content" in event:
                parts = event["content"].get("parts", [])
                for part in parts:
                    if isinstance(part, dict) and "text" in part:
                        print(part["text"], end="")
            elif hasattr(event, "content") and hasattr(event.content, "parts"):
                for part in event.content.parts:
                    if hasattr(part, "text"):
                        print(part.text, end="")
        print("\n" + "-" * 50)

# Run test execution across multiple US cities
test_cities = ["Miami, FL", "Dallas, TX", "Seattle, WA"]

# Execute tests on Gemini Agent
test_weather_agent(weather_agent_gemini, test_cities)

# Execute tests on GPT-4o Agent
# test_weather_agent(weather_agent_gpt, test_cities)

This legacy setting overrides the new Cloud Console toggle and environment variable controls.
Impact: The Cloud Console may incorrectly show telemetry as 'On' when it is actually 'Off', and the UI toggle will not work.
Action: To fix this and control telemetry, please remove the 'enable_tracing' parameter from your deployment code.
You can then use the 'GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY' environment variable:
agent_engines.create(
  env_vars={
    "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": true|false
  }
)
or the toggle in the Cloud Console: https://console.cloud.google.com/vertex-ai/agents.



================ TESTING AGENT: Pat_Gemini ================

--- Location Check: Miami, FL ---
Weather condition alerts for Miami, FL are currently CLEAR.
--------------------------------------------------

--- Location Check: Dallas, TX ---
There is an active **Extreme Heat Warning** in Dallas, TX.

**Severity:** Severe
**Urgency:** Expected

**Safety Steps:**
*   Stay hydrated and wear lightweight, loose-fitting clothing.
*   Limit strenuous activities to early morning or evening.
*   Check on relatives and neighbors.
*   Never leave children or pets in enclosed vehicles.
*   Be aware of heat exhaustion and heat stroke symptoms. If someone is overcome by heat, move them to a cool, shaded area and call 911 for heat stroke.
*   Limit outdoor time, especially from 2 PM to 7 PM, and use cooling centers if available.
--------------------------------------------------

--- Location Check: Seattle, WA ---
Weather condition alerts for Seattle, WA are currently CLEAR.
-----------------------